# Speech Emotion Recognition — CNN-LSTM

## 1. Install & Import Dependencies

In [ ]:
!pip install librosa audioread soundfile -q

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch: {torch.__version__}')

# Device selection — supports CUDA, Apple Silicon MPS, and CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

## 2. Configuration

In [ ]:
# ── SET THIS TO YOUR LOCAL DATASET FOLDER ─────────────────────────────────────
DRIVE_ROOT = '/Users/isaaccampbell/datasets/emotion_audio'  # <-- update this
# ──────────────────────────────────────────────────────────────────────────────

METADATA_CSV     = os.path.join(DRIVE_ROOT, 'combined_metadata.csv')
CACHE_PATH       = 'features_cache.pkl'
BEST_MODEL_PATH  = 'best_emotion_cnnlstm.pt'

# Audio config
SAMPLE_RATE  = 22050
DURATION     = 3.0
N_MFCC       = 40
N_MELS       = 128
HOP_LENGTH   = 512
N_FFT        = 2048

# Training config
BATCH_SIZE   = 32
EPOCHS       = 150
LR           = 3e-4
RANDOM_SEED  = 42

# Emotions to keep (calm -> neutral, surprised dropped)
EMOTION_MAP  = {'calm': 'neutral'}
DROP_EMOTIONS = ['surprised']

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print('Config ready.')

## 3. Load & Clean Metadata

In [ ]:
df = pd.read_csv(METADATA_CSV)

# Keep only valid files
df = df[df['is_valid'] == True].reset_index(drop=True)

# Merge calm -> neutral, drop surprised
df['emotion'] = df['emotion'].replace(EMOTION_MAP)
df = df[~df['emotion'].isin(DROP_EMOTIONS)].reset_index(drop=True)

# Build absolute file paths
df['full_path'] = df['filepath'].apply(lambda p: os.path.join(DRIVE_ROOT, p))

print(f'Total samples: {len(df)}')
print(f'\nEmotion distribution:')
print(df['emotion'].value_counts())
print(f'\nSources:')
print(df['source'].value_counts())

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Emotion counts
emotion_counts = df['emotion'].value_counts()
axes[0].bar(emotion_counts.index, emotion_counts.values,
            color=plt.cm.Set2.colors[:len(emotion_counts)])
axes[0].set_title('Samples per Emotion', fontsize=14)
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Gender breakdown
gender_emotion = df.groupby(['emotion', 'gender']).size().unstack(fill_value=0)
gender_emotion.plot(kind='bar', ax=axes[1], colormap='Set1')
axes[1].set_title('Emotion by Gender', fontsize=14)
axes[1].tick_params(axis='x', rotation=30)

# Duration distribution
axes[2].hist(df['duration_sec'], bins=40, color='steelblue', edgecolor='white')
axes[2].axvline(DURATION, color='red', linestyle='--', label=f'Clip length ({DURATION}s)')
axes[2].set_title('Audio Duration Distribution', fontsize=14)
axes[2].set_xlabel('Duration (seconds)')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample waveform + spectrogram for each emotion
emotions = sorted(df['emotion'].unique())
fig, axes = plt.subplots(len(emotions), 2, figsize=(16, len(emotions) * 2.5))

for i, emotion in enumerate(emotions):
    sample = df[df['emotion'] == emotion].iloc[0]
    try:
        y, sr = librosa.load(sample['full_path'], sr=SAMPLE_RATE, duration=DURATION)
        y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * DURATION))
        librosa.display.waveshow(y, sr=sr, ax=axes[i, 0], color='steelblue')
        axes[i, 0].set_title(f'{emotion.upper()} — Waveform')
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                              hop_length=HOP_LENGTH, n_fft=N_FFT)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, x_axis='time', y_axis='mel',
                                        sr=sr, hop_length=HOP_LENGTH,
                                        ax=axes[i, 1], cmap='magma')
        axes[i, 1].set_title(f'{emotion.upper()} — Log-Mel Spectrogram')
        fig.colorbar(img, ax=axes[i, 1], format='%+2.0f dB')
    except Exception as e:
        axes[i, 0].text(0.5, 0.5, f'Error: {e}', ha='center', va='center')

plt.tight_layout()
plt.savefig('emotion_spectrograms.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Feature Extraction

Features are cached to disk after the first run — subsequent runs load in seconds.

In [ ]:
def load_audio(path, sr=SAMPLE_RATE, duration=DURATION):
    y, _ = librosa.load(path, sr=sr, duration=duration)
    y = librosa.util.fix_length(y, size=int(sr * duration))
    return y


def extract_handcrafted_features(y, sr=SAMPLE_RATE):
    """Statistical summary features for the MLP baseline."""
    features = []

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    delta_mfcc = librosa.feature.delta(mfcc)
    features.extend(np.mean(delta_mfcc, axis=1))

    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64,
                                          hop_length=HOP_LENGTH, n_fft=N_FFT)
    features.extend(np.mean(mel, axis=1))

    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
    features.append(float(np.mean(zcr)))
    features.append(float(np.std(zcr)))

    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    features.append(float(np.mean(rms)))
    features.append(float(np.std(rms)))

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.append(float(np.mean(centroid)))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.append(float(np.mean(rolloff)))

    return np.array(features, dtype=np.float32)


def extract_log_mel(y, sr=SAMPLE_RATE):
    """2D log-mel spectrogram for CNN-LSTM input."""
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                          hop_length=HOP_LENGTH, n_fft=N_FFT)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)


print('Feature functions defined.')

In [ ]:
from tqdm import tqdm

if os.path.exists(CACHE_PATH):
    print(f'Loading features from cache: {CACHE_PATH}')
    with open(CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    X_hc     = cache['X_hc']
    X_mel    = cache['X_mel']
    y_labels = cache['labels']
    print(f'Loaded {len(y_labels)} samples.')
else:
    print(f'Extracting features from {len(df)} files...')
    handcrafted_features, log_mel_features, labels = [], [], []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            y = load_audio(row['full_path'])
            handcrafted_features.append(extract_handcrafted_features(y))
            log_mel_features.append(extract_log_mel(y))
            labels.append(row['emotion'])
        except Exception as e:
            print(f'  Skipped {row["filename"]}: {e}')

    X_hc     = np.array(handcrafted_features)
    X_mel    = np.array(log_mel_features)
    y_labels = np.array(labels)

    with open(CACHE_PATH, 'wb') as f:
        pickle.dump({'X_hc': X_hc, 'X_mel': X_mel, 'labels': y_labels}, f)
    print(f'Features cached to {CACHE_PATH}')

# Apply emotion mapping to cached labels in case cache was built before merging
y_labels = np.array([EMOTION_MAP.get(l, l) for l in y_labels])
mask = np.array([l not in DROP_EMOTIONS for l in y_labels])
X_hc, X_mel, y_labels = X_hc[mask], X_mel[mask], y_labels[mask]

print(f'\nFinal dataset: {len(y_labels)} samples')
print(f'Handcrafted shape : {X_hc.shape}')
print(f'Log-mel shape     : {X_mel.shape}')
print(f'Label distribution: {Counter(y_labels)}')

## 6. Encode Labels, Split & Oversample

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)
NUM_CLASSES = len(le.classes_)
print(f'Classes ({NUM_CLASSES}): {le.classes_}')

# 70 / 15 / 15 split
X_hc_train, X_hc_tmp, X_mel_train, X_mel_tmp, y_train, y_tmp = train_test_split(
    X_hc, X_mel, y_encoded, test_size=0.30,
    random_state=RANDOM_SEED, stratify=y_encoded
)
X_hc_val, X_hc_test, X_mel_val, X_mel_test, y_val, y_test = train_test_split(
    X_hc_tmp, X_mel_tmp, y_tmp, test_size=0.50,
    random_state=RANDOM_SEED, stratify=y_tmp
)

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

In [ ]:
# Oversample training set so all classes are equal
max_count = max(Counter(y_train).values())
X_mel_parts, X_hc_parts, y_parts = [], [], []

for cls in np.unique(y_train):
    mask = y_train == cls
    X_mel_res, X_hc_res, y_res = resample(
        X_mel_train[mask], X_hc_train[mask], y_train[mask],
        n_samples=max_count, random_state=RANDOM_SEED
    )
    X_mel_parts.append(X_mel_res)
    X_hc_parts.append(X_hc_res)
    y_parts.append(y_res)

X_mel_train = np.vstack(X_mel_parts)
X_hc_train  = np.vstack(X_hc_parts)
y_train     = np.concatenate(y_parts)

print(f'Balanced training set: {len(y_train)} samples')
print(Counter(y_train))

# Class weights (still useful for loss function)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'\nClass weights: {dict(zip(le.classes_, class_weights.round(3)))}')

## 7. Baseline: MLP Classifier

In [ ]:
scaler = StandardScaler()
X_hc_train_s = scaler.fit_transform(X_hc_train)
X_hc_val_s   = scaler.transform(X_hc_val)
X_hc_test_s  = scaler.transform(X_hc_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_SEED,
    verbose=False
)
mlp.fit(X_hc_train_s, y_train)

mlp_val_acc  = mlp.score(X_hc_val_s, y_val)
mlp_test_acc = mlp.score(X_hc_test_s, y_test)
print(f'MLP Val Accuracy : {mlp_val_acc:.4f}')
print(f'MLP Test Accuracy: {mlp_test_acc:.4f}')

y_pred_mlp = mlp.predict(X_hc_test_s)
print('\nMLP Classification Report:')
print(classification_report(y_test, y_pred_mlp, target_names=le.classes_))

## 8. Main Model: CNN-LSTM

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, X_mel, y, augment=False):
        self.X = torch.FloatTensor(X_mel).unsqueeze(1)  # (N, 1, N_MELS, T)
        self.y = torch.LongTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            x = self._augment(x)
        return x, self.y[idx]

    def _augment(self, x):
        """SpecAugment with reduced mask sizes to avoid over-regularizing."""
        x = x.clone()
        _, n_mels, time_steps = x.shape

        # Frequency masking
        f = np.random.randint(0, max(1, n_mels // 12))
        f0 = np.random.randint(0, n_mels - f)
        x[:, f0:f0+f, :] = 0

        # Time masking
        t = np.random.randint(0, max(1, time_steps // 12))
        t0 = np.random.randint(0, time_steps - t)
        x[:, :, t0:t0+t] = 0

        return x


train_ds = EmotionDataset(X_mel_train, y_train, augment=True)
val_ds   = EmotionDataset(X_mel_val,   y_val,   augment=False)
test_ds  = EmotionDataset(X_mel_test,  y_test,  augment=False)

# num_workers=0 for Mac compatibility
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_x, _ = next(iter(train_loader))
print(f'Batch shape: {sample_x.shape}')  # (B, 1, N_MELS, T)

In [ ]:
class EmotionCNNLSTM(nn.Module):
    """
    CNN-LSTM for speech emotion recognition.

    CNN blocks extract local spectral features from each time frame.
    Frequency axis is pooled down; time axis is preserved.
    Bidirectional LSTM then models how features evolve over time.
    """
    def __init__(self, num_classes, lstm_hidden=256, lstm_layers=2):
        super().__init__()

        # CNN feature extractor
        # Pool (2,1) — only pool frequency axis, keep full time dimension for LSTM
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),   # freq: 128 -> 64
            nn.Dropout2d(0.25),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),   # freq: 64 -> 32
            nn.Dropout2d(0.25),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),   # freq: 32 -> 16
            nn.Dropout2d(0.25),
        )

        # After 3 x (2,1) pools on 128 mel bands: 128 / 8 = 16 freq bins
        # CNN output per time step = 128 channels * 16 freq bins = 2048
        cnn_out_features = 128 * 16

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=cnn_out_features,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 256),  # *2 for bidirectional
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (batch, 1, n_mels, time)
        x = self.cnn(x)
        # x: (batch, channels, freq, time)

        batch, channels, freq, time = x.shape
        # Reshape to (batch, time, channels * freq) for LSTM
        x = x.permute(0, 3, 1, 2).reshape(batch, time, channels * freq)

        # LSTM — take output of last time step
        x, _ = self.lstm(x)
        x = x[:, -1, :]

        return self.classifier(x)


model = EmotionCNNLSTM(NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'CNN-LSTM parameters: {total_params:,}')
print(model)

## 9. Train CNN-LSTM

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += len(yb)
    return total_loss / total, correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

print(f'Training CNN-LSTM for {EPOCHS} epochs on {DEVICE}...\n')

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    if epoch % 10 == 0 or epoch == 1:
        best_flag = ' ✅ best' if vl_acc == best_val_acc else ''
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | '
              f'Val Loss: {vl_loss:.4f}, Acc: {vl_acc:.4f}{best_flag}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

## 10.Evaluate

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('CNN-LSTM Training History', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# Load best weights
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb    = xb.to(DEVICE)
        logits = model(xb)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())
        all_probs.extend(probs)

cnn_test_acc = np.mean(np.array(all_preds) == np.array(all_true))
print(f'CNN-LSTM Test Accuracy : {cnn_test_acc:.4f}')
print(f'MLP     Test Accuracy  : {mlp_test_acc:.4f}  (baseline)')
print()
print('CNN-LSTM Classification Report:')
print(classification_report(all_true, all_preds, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('CNN-LSTM Confusion Matrix (counts)', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title('CNN-LSTM Confusion Matrix (normalized)', fontsize=13)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('cnnlstm_confusion_matrix.png', dpi=150)
plt.show()

## 12. Save Models

In [ ]:
# Save CNN-LSTM with full config
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': NUM_CLASSES,
    'class_labels': list(le.classes_),
    'config': {
        'sample_rate': SAMPLE_RATE,
        'duration': DURATION,
        'n_mels': N_MELS,
        'hop_length': HOP_LENGTH,
        'n_fft': N_FFT
    }
}, 'emotion_cnnlstm_final.pt')

# Save MLP + scaler + label encoder
with open('emotion_mlp_artifacts.pkl', 'wb') as f:
    pickle.dump({'mlp': mlp, 'scaler': scaler, 'label_encoder': le}, f)

print('Saved:')
print('  emotion_cnnlstm_final.pt')
print('  emotion_mlp_artifacts.pkl')